# Airbnb Price Analysis — New York City

**What actually drives Airbnb prices? And how much does location really matter?**

You can find two seemingly identical listings a few blocks apart — same room type, similar reviews — but one charges \$90/night and the other \$400. Location is the obvious culprit, but let's actually *measure* it.

We'll use the [NYC Airbnb Open Data](https://www.kaggle.com/dgomonov/new-york-city-airbnb-open-data) dataset — 48,895 listings scraped in 2019. It covers all five boroughs and has just enough features to tell an interesting story without being overwhelming.

---

**Roadmap:**
1. Load & clean the data
2. Understand the price distribution (spoiler: it's very skewed)
3. Break down prices by room type, reviews, and availability
4. Deep-dive into location — borough → neighbourhood → raw coordinates
5. Train a Random Forest and let it tell us what matters most

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Global plot style
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})
sns.set_palette('husl')

print("Libraries loaded ✓")

## 1. Load the Data

In [ ]:
# Two mirrors of the same dataset — use whichever loads
URLS = [
    "https://raw.githubusercontent.com/pjournal/boun01g-data-mine-r-s/gh-pages/Assignment/AB_NYC_2019.csv",
    "https://raw.githubusercontent.com/ManarOmar/New-York-Airbnb-2019/master/AB_NYC_2019.csv",
]

df = None
for url in URLS:
    try:
        df = pd.read_csv(url)
        print(f"Loaded {len(df):,} rows from:\n  {url}")
        break
    except Exception as e:
        print(f"Failed: {e}")

if df is None:
    raise RuntimeError("Could not load data. Download AB_NYC_2019.csv from Kaggle and place it in this folder.")

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}\n")
df.head()

In [ ]:
# Missing value audit
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'count': missing, 'pct': missing_pct}).query('count > 0').sort_values('pct', ascending=False)

Only `reviews_per_month` and `last_review` have missing values — both come from listings with zero reviews. That's expected and not a problem for our analysis.

## 2. Cleaning Up

In [ ]:
# Drop the tiny handful of rows with no price
df = df.dropna(subset=['price'])
df = df[df['price'] > 0]  # zero-price = data error or 'contact host'

# Fill zero reviews_per_month with 0 (not NaN)
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# Quick summary
print(f"Working dataset: {len(df):,} listings")
print(f"\nPrice summary:")
print(df['price'].describe().apply(lambda x: f'${x:,.0f}'))

The max price of \$10,000 is almost certainly real (luxury penthouses), but it'll squish our charts. We'll keep the full data for modeling but filter to ≤\$500 for most visualizations — that covers ~97% of listings and makes the plots readable.

In [ ]:
# Plotting subset — exclude extreme outliers
df_plot = df[df['price'] <= 500].copy()
pct_kept = len(df_plot) / len(df) * 100
print(f"Plotting on {len(df_plot):,} listings ({pct_kept:.1f}% of data, price ≤ $500)")

## 3. Price Distribution — It's Very Skewed

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw price
axes[0].hist(df_plot['price'], bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_title('Price Distribution (raw)')
axes[0].set_xlabel('Price per night ($)')
axes[0].set_ylabel('Number of listings')
axes[0].axvline(df_plot['price'].median(), color='tomato', linestyle='--', label=f'Median: ${df_plot["price"].median():.0f}')
axes[0].axvline(df_plot['price'].mean(), color='orange', linestyle='--', label=f'Mean: ${df_plot["price"].mean():.0f}')
axes[0].legend()

# Log-transformed price
axes[1].hist(np.log1p(df_plot['price']), bins=60, color='mediumseagreen', edgecolor='white', linewidth=0.4)
axes[1].set_title('Price Distribution (log scale)')
axes[1].set_xlabel('log(price + 1)')
axes[1].set_ylabel('Number of listings')

plt.suptitle('Airbnb Nightly Prices — NYC 2019', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Skewness: {df_plot['price'].skew():.2f}  (0 = symmetric, >1 = right-skewed)")

Classic long-tail distribution. The mean (\$152) is pulled far above the median (\$106) by expensive outliers. The log-transformed version looks almost normal — which tells us price behaves *multiplicatively* (percentage differences are more meaningful than absolute dollar differences). We'll keep this in mind for modeling.

## 4. Room Type: The Biggest Filter

In [ ]:
room_stats = df.groupby('room_type')['price'].agg(['median', 'mean', 'count']).sort_values('median', ascending=False)
room_stats.columns = ['Median Price', 'Mean Price', 'Count']
room_stats['Share (%)'] = (room_stats['Count'] / room_stats['Count'].sum() * 100).round(1)
print(room_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Median price bar chart
room_order = df.groupby('room_type')['price'].median().sort_values(ascending=False).index
colors = ['#2196F3', '#4CAF50', '#FF9800']

medians = df.groupby('room_type')['price'].median().reindex(room_order)
bars = axes[0].bar(room_order, medians, color=colors, width=0.55)
axes[0].set_title('Median Price by Room Type')
axes[0].set_xlabel('')
axes[0].set_ylabel('Median price per night ($)')
for bar, val in zip(bars, medians):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, f'${val:.0f}', ha='center', fontweight='bold')

# Box plot (capped at $500 for readability)
bp_data = [df_plot[df_plot['room_type'] == rt]['price'].values for rt in room_order]
bp = axes[1].boxplot(bp_data, labels=room_order, patch_artist=True, medianprops={'color': 'black', 'linewidth': 2})
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Price Spread by Room Type (capped $500)')
axes[1].set_ylabel('Price per night ($)')

plt.tight_layout()
plt.show()

Room type is probably the single most powerful variable: **entire homes cost ~3× more than private rooms** and private rooms cost ~2× more than shared rooms. The spread within each category is enormous though — a cheap entire apartment and an expensive private room can easily overlap. So room type is a big factor but not the whole story.

## 5. Reviews: Quantity vs. Price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bin number_of_reviews and compute median price
df_r = df_plot.copy()
df_r['review_bucket'] = pd.cut(
    df_r['number_of_reviews'],
    bins=[0, 0, 5, 20, 50, 100, 700],
    labels=['0', '1–5', '6–20', '21–50', '51–100', '100+'],
    include_lowest=True
)
review_price = df_r.groupby('review_bucket')['price'].median()

axes[0].bar(review_price.index, review_price.values, color='mediumpurple', alpha=0.8)
axes[0].set_title('Median Price by Number of Reviews')
axes[0].set_xlabel('Number of reviews')
axes[0].set_ylabel('Median price ($)')

# Reviews per month vs price scatter (sample)
sample = df_plot[df_plot['reviews_per_month'] <= 10].sample(3000, random_state=42)
axes[1].scatter(sample['reviews_per_month'], sample['price'], alpha=0.2, s=15, color='mediumpurple')
axes[1].set_title('Reviews per Month vs. Price')
axes[1].set_xlabel('Reviews per month')
axes[1].set_ylabel('Price ($)')
corr = sample[['reviews_per_month', 'price']].corr().iloc[0, 1]
axes[1].text(0.65, 0.9, f'r = {corr:.3f}', transform=axes[1].transAxes, fontsize=11)

plt.tight_layout()
plt.show()

Interesting pattern: **listings with zero reviews are pricier on average**, then price drops as review count increases, before rising slightly for very high-review listings. Two competing forces here:
- New/premium listings tend to start at higher prices
- High-volume popular listings are often mid-range priced (volume strategy)
- The most-reviewed listings have some prestige and can command a premium

Reviews per month has a very weak negative correlation with price (r ≈ −0.05). Review count alone is a poor price predictor.

## 6. Availability: Do Pricier Listings Sit Empty?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Availability buckets
df_a = df_plot.copy()
df_a['avail_bucket'] = pd.cut(
    df_a['availability_365'],
    bins=[-1, 0, 90, 180, 270, 365],
    labels=['Blocked\n(0)', 'Low\n(1–90)', 'Medium\n(91–180)', 'High\n(181–270)', 'Very High\n(271–365)']
)
avail_price = df_a.groupby('avail_bucket')['price'].median()
avail_count = df_a['avail_bucket'].value_counts().sort_index()

axes[0].bar(avail_price.index, avail_price.values, color='darkcyan', alpha=0.8)
axes[0].set_title('Median Price by Availability')
axes[0].set_xlabel('Availability window (days/year)')
axes[0].set_ylabel('Median price ($)')

axes[1].bar(avail_count.index, avail_count.values, color='darkcyan', alpha=0.5)
axes[1].set_title('Listing Count by Availability')
axes[1].set_xlabel('Availability window (days/year)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

Blocked listings (availability = 0) tend to be pricier — makes sense, they're probably in high-demand areas and get booked out quickly. Very highly available listings (almost always open) tend to be cheaper — possibly harder to fill at premium prices.

## 7. Location: Borough Level

In [ ]:
borough_stats = df.groupby('neighbourhood_group')['price'].agg(
    count='count',
    median='median',
    mean='mean',
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75)
).sort_values('median', ascending=False)

print(borough_stats.round(0).to_string())

In [ ]:
borough_order = df.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False).index
borough_palette = {'Manhattan': '#E53935', 'Brooklyn': '#FB8C00', 'Queens': '#43A047', 'Bronx': '#1E88E5', 'Staten Island': '#8E24AA'}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Median bar chart
medians = df.groupby('neighbourhood_group')['price'].median().reindex(borough_order)
bars = axes[0].bar(
    borough_order, medians,
    color=[borough_palette[b] for b in borough_order],
    width=0.55, alpha=0.85
)
axes[0].set_title('Median Nightly Price by Borough')
axes[0].set_ylabel('Median price ($)')
for bar, val in zip(bars, medians):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                 f'${val:.0f}', ha='center', fontsize=11, fontweight='bold')

# Violin plot for full distribution view
data_by_borough = [df_plot[df_plot['neighbourhood_group'] == b]['price'].values for b in borough_order]
vp = axes[1].violinplot(data_by_borough, showmedians=True, showextrema=False)
for body, b in zip(vp['bodies'], borough_order):
    body.set_facecolor(borough_palette[b])
    body.set_alpha(0.7)
vp['cmedians'].set_color('black')
vp['cmedians'].set_linewidth(2)
axes[1].set_xticks(range(1, len(borough_order) + 1))
axes[1].set_xticklabels(borough_order)
axes[1].set_title('Price Distribution by Borough (capped $500)')
axes[1].set_ylabel('Price ($)')

plt.suptitle('Location Shapes the Price Floor — Borough Level', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Manhattan is in a different league.** Its median of ~\$150 is nearly 50% higher than Brooklyn (~\$90) and over 2× the Bronx (~\$65). But look at the violin shapes — every borough has a long upper tail, meaning expensive listings exist everywhere. The *floor* shifts by location; the ceiling doesn't.

## 8. Location: Neighbourhood Level

In [ ]:
# Need at least 30 listings to trust the median
neighbourhood_stats = (
    df.groupby(['neighbourhood_group', 'neighbourhood'])['price']
    .agg(count='count', median='median', mean='mean')
    .query('count >= 30')
    .sort_values('median', ascending=False)
)

print(f"Neighbourhoods with ≥30 listings: {len(neighbourhood_stats)}")
print("\nTop 10 most expensive:")
print(neighbourhood_stats.head(10).to_string())

In [ ]:
top20 = neighbourhood_stats.head(20).reset_index()
bottom20 = neighbourhood_stats.tail(20).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, data, title, default_color in [
    (axes[0], top20[::-1], 'Top 20 Most Expensive Neighbourhoods', 'tomato'),
    (axes[1], bottom20[::-1], 'Top 20 Most Affordable Neighbourhoods', 'steelblue')
]:
    colors = [borough_palette.get(b, default_color) for b in data['neighbourhood_group']]
    bars = ax.barh(data['neighbourhood'], data['median'], color=colors, alpha=0.85, height=0.65)
    ax.set_xlabel('Median price per night ($)')
    ax.set_title(title, fontsize=12)
    for bar, val, borough in zip(bars, data['median'], data['neighbourhood_group']):
        ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2,
                f'${val:.0f}  ({borough[:3]})', va='center', fontsize=8)
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='y', length=0)

# Borough legend
from matplotlib.patches import Patch
legend_patches = [Patch(color=c, label=b) for b, c in borough_palette.items()]
fig.legend(handles=legend_patches, title='Borough', loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Neighbourhood-Level Price Gap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

The spread is striking. The priciest neighbourhoods (think Fort Wadsworth, Tribeca, NoHo) have a median above \$200 — while the most affordable are under \$50. That's a 4×+ difference from neighbourhood choice alone, within the same city. This is what we mean when we say location dominates price.

## 9. Geographic Price Map

In [ ]:
# Sample for performance — 10k points gives a good visual without being slow
map_df = df[df['price'] <= 500].sample(min(10000, len(df)), random_state=42)

fig = px.scatter_mapbox(
    map_df,
    lat='latitude',
    lon='longitude',
    color='price',
    color_continuous_scale='Plasma',
    range_color=[0, 300],
    size='price',
    size_max=12,
    opacity=0.65,
    zoom=9.5,
    center={'lat': 40.7128, 'lon': -74.006},
    mapbox_style='open-street-map',
    hover_data={'price': True, 'neighbourhood': True, 'room_type': True, 'latitude': False, 'longitude': False},
    title='Airbnb Listings in NYC — Colored by Nightly Price',
    labels={'price': 'Price ($)'}
)

fig.update_layout(
    height=600,
    margin={'r': 0, 't': 40, 'l': 0, 'b': 0},
    coloraxis_colorbar=dict(title='Price ($)', tickprefix='$')
)
fig.show()

In [ ]:
# Density heatmap — where do listings cluster, and are those areas expensive?
fig = px.density_mapbox(
    map_df,
    lat='latitude',
    lon='longitude',
    z='price',
    radius=10,
    zoom=9.5,
    center={'lat': 40.7128, 'lon': -74.006},
    mapbox_style='open-street-map',
    color_continuous_scale='Hot',
    title='Price Density Map — Brighter = Higher Price Concentration',
    labels={'z': 'Price ($)'}
)

fig.update_layout(
    height=600,
    margin={'r': 0, 't': 40, 'l': 0, 'b': 0}
)
fig.show()

The scatter map makes the Manhattan premium visually obvious — dense cluster of bright (expensive) dots in Midtown, Lower Manhattan, and the West Village. Brooklyn has a warm hotspot in Williamsburg/DUMBO. Queens and the Bronx are mostly cool-toned (affordable).

One interesting feature: **you can see the subway lines** in the listing density map. Hosts know that transit access = bookings.

## 10. Correlation Overview

In [ ]:
# Encode categorical variables for correlation
df_corr = df_plot.copy()
le = LabelEncoder()
df_corr['room_type_enc'] = le.fit_transform(df_corr['room_type'])
df_corr['borough_enc'] = le.fit_transform(df_corr['neighbourhood_group'])

corr_cols = ['price', 'room_type_enc', 'borough_enc', 'latitude', 'longitude',
             'minimum_nights', 'number_of_reviews', 'reviews_per_month',
             'calculated_host_listings_count', 'availability_365']

corr_labels = ['price', 'room type', 'borough', 'latitude', 'longitude',
               'min nights', 'n reviews', 'reviews/mo',
               'host listings', 'availability']

corr_matrix = df_corr[corr_cols].corr()
corr_matrix.index = corr_labels
corr_matrix.columns = corr_labels

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlBu_r', vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Pearson r'}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Most correlations with price are weak — which is actually *expected*. If price were highly correlated with a single feature, pricing would be trivial. What we're seeing is that price is driven by a **combination of factors**, not any one of them alone. This is exactly where a Random Forest model shines — it captures non-linear interactions between features.

## 11. What Actually Drives Price? — Random Forest Feature Importance

In [ ]:
# Feature engineering
df_model = df.copy()

# Encode categorical features
le_room = LabelEncoder()
le_nb = LabelEncoder()
le_nbg = LabelEncoder()

df_model['room_type_enc'] = le_room.fit_transform(df_model['room_type'])
df_model['neighbourhood_enc'] = le_nb.fit_transform(df_model['neighbourhood'])
df_model['borough_enc'] = le_nbg.fit_transform(df_model['neighbourhood_group'])

# Features (exclude raw categoricals and id columns)
feature_cols = [
    'latitude', 'longitude',
    'room_type_enc', 'borough_enc', 'neighbourhood_enc',
    'minimum_nights', 'number_of_reviews', 'reviews_per_month',
    'calculated_host_listings_count', 'availability_365'
]

feature_labels = [
    'Latitude', 'Longitude',
    'Room Type', 'Borough', 'Neighbourhood',
    'Min Nights', 'Num Reviews', 'Reviews/Month',
    'Host Listings Count', 'Availability (days/yr)'
]

X = df_model[feature_cols].fillna(0)
y = np.log1p(df_model['price'])  # predict log-price (better for skewed targets)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train):,} rows")
print(f"Test set:     {len(X_test):,} rows")

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Model Performance:")
print(f"  R²  = {r2:.3f}  (1.0 = perfect, 0 = predicts mean every time)")
print(f"  MAE = ${mae:.2f} per night  (on original price scale)")

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_labels).sort_values(ascending=True)

# Color: location features vs others
location_features = {'Latitude', 'Longitude', 'Borough', 'Neighbourhood'}
colors = ['#E53935' if f in location_features else '#1E88E5' for f in importances.index]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(importances.index, importances.values, color=colors, alpha=0.85, height=0.65)
ax.set_xlabel('Feature Importance (mean decrease in impurity)')
ax.set_title('What Drives Airbnb Price? — Random Forest Feature Importance', fontsize=14, fontweight='bold')

# Value labels
for bar, val in zip(bars, importances.values):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E53935', alpha=0.85, label='Location features'),
    Patch(facecolor='#1E88E5', alpha=0.85, label='Non-location features')
]
ax.legend(handles=legend_elements, loc='lower right')
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)

plt.tight_layout()
plt.show()

# Summarize contribution
location_imp = importances[importances.index.isin(location_features)].sum()
other_imp = importances[~importances.index.isin(location_features)].sum()
print(f"\nLocation features combined importance: {location_imp:.1%}")
print(f"All other features combined:          {other_imp:.1%}")

In [ ]:
# Actual vs predicted scatter
fig, ax = plt.subplots(figsize=(8, 6))
sample_idx = np.random.choice(len(y_test), 2000, replace=False)
actual = np.expm1(y_test.values[sample_idx])
predicted = np.expm1(y_pred[sample_idx])

ax.scatter(actual, predicted, alpha=0.15, s=15, color='steelblue')
max_val = min(600, max(actual.max(), predicted.max()))
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual price ($)')
ax.set_ylabel('Predicted price ($)')
ax.set_title(f'Actual vs. Predicted Prices  (R² = {r2:.3f}, MAE = ${mae:.0f})', fontsize=12)
ax.set_xlim(0, 600)
ax.set_ylim(0, 600)
ax.legend()
plt.tight_layout()
plt.show()

## 12. Conclusions

### What drives Airbnb prices in NYC?

| Factor | Impact | Notes |
|--------|--------|-------|
| **Room type** | Very high | Entire home ≈ 3× private room ≈ 5× shared room |
| **Location (neighbourhood)** | Very high | Up to 4× difference within the city |
| **Borough** | High | Manhattan median ≈ 2× the Bronx |
| **Coordinates (lat/lon)** | High | Even within neighbourhoods, exact location matters |
| **Host listing count** | Moderate | Professional hosts list differently |
| **Availability** | Moderate | Low availability = high demand = higher price |
| **Reviews** | Low | Weak signal — popular ≠ expensive |
| **Minimum nights** | Low | Mostly a host preference signal |

### Key takeaways

1. **Location explains the majority of price variance.** When we add up latitude, longitude, neighbourhood, and borough, these location features account for the largest share of the Random Forest's predictive power. A listing's ZIP code is often more predictive of its price than its amenities.

2. **Room type is the dominant non-location signal.** The structural difference between renting a whole apartment vs. a spare room matters more than any other feature we measured.

3. **Price is multiplicative, not additive.** The log-normal distribution of prices means you should think in percentages, not dollars. Manhattan doesn't charge \$60 *more* per night — it charges 80% *more*.

4. **Reviews are surprisingly weak predictors.** More reviews ≠ higher price. If anything, high-review-count listings tend to be mid-range (volume strategy).

5. **The model is decent but not perfect (R² ≈ 0.60).** The remaining ~40% unexplained variance likely comes from listing quality, amenities, photos, and host reputation — things not captured in this dataset.

### What's missing?

To build a production-quality pricing model you'd want:
- **Amenity data** (WiFi, parking, kitchen, pool)
- **Photo quality scores** (a huge driver of perceived value)
- **Distance to landmarks** (Central Park, subway stations, airports)
- **Seasonal/temporal patterns** (this dataset is a single snapshot)
- **Neighbourhood trend data** (is the area gentrifying?)